# Snow Cover Fraction (SCF) and Slope Data Processing for HUC12s
This notebook retrieves Snow Cover Fraction + SWE values from the ERA5 - Land dataset. It also retrieves slope data USGS 3DEP 10m digital elevation model (DEM) using Google Earth Engine. 
The worklfow involves:
- Downloading and merging HUC12 GeoJSONs from S3, creating shapefiles for GEE asset upload.
- Checking coverage of HUC12s in DHSVM output and fetching missing boundaries from USGS NHD API.
- Exporting yearwise ERA5 snow cover and SWE data for HUC12s using GEE batch export.
- Extracting slope statistics for HUC12s from DEM data in GEE.
- Loading ERA5 and slope data, merging with model-ready HUC CSVs, and uploading enriched files to S3.



In [ ]:
import boto3
import io
import ee
from botocore.exceptions import ClientError
import json
import geopandas as gpd
import pandas as pd
import numpy as np
import zipfile
import requests
from shapely.geometry import shape

from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive

ee.Authenticate(auth_mode='notebook')
ee.Initialize(project='ee-riyoshas')

s3_client = boto3.client('s3', region_name='us-west-2')

SHAPE_BUCKET = 'snowml-shape'
MODEL_READY_BUCKET = 'snowml-model-ready'

# EE Asset ID
ASSET_ID = 'projects/ee-riyoshas/assets/huc12_boundaries'


In [2]:
def download_geojson_from_s3(bucket, key):
    """Download GeoJSON from S3 bucket"""
    try:
        print(f"Downloading {key} from {bucket}...")
        response = s3_client.get_object(Bucket=bucket, Key=key)
        geojson_data = json.loads(response['Body'].read().decode('utf-8'))
        print(f"Successfully downloaded GeoJSON")
        return geojson_data
    except ClientError as e:
        print(f"Error downloading from S3: {e}")
        return None

def load_huc12_data(bucket, key):
    """Load HUC12 data as GeoDataFrame"""
    geojson_data = download_geojson_from_s3(bucket, key)
    if geojson_data:
        gdf = gpd.GeoDataFrame.from_features(geojson_data['features'])
        gdf.set_crs(epsg=4326, inplace=True)
        return gdf
    return None

We create a `huc12_boundaries.shp` file to upload to Google Earth Engine Assets, so that HUC12 boundaries can easily be accessed. This has to be manually uploaded.

In [ ]:
gdfs = []
all_huc12s = [
    # PNW coordinates
 'Huc12_in_17020009.geojson',
 'Huc12_in_17020011.geojson',
 'Huc12_in_17110005.geojson',
 'Huc12_in_17110006.geojson',
 'Huc12_in_17110009.geojson',
 'Huc12_in_17010304.geojson',
 'Huc12_in_17010302.geojson',
 'Huc12_in_17060207.geojson',
 'Huc12_in_17060208.geojson',
 'Huc12_in_17030001.geojson',
 'Huc12_in_17030002.geojson',
 'Huc12_in_17110008.geojson',
 'Huc12_in_17030003.geojson',
 'Huc12_in_17020010.geojson',
 'Huc12_in_17110007.geojson',
]             
for HUC12_KEY in all_huc12s:
    gdf = load_huc12_data(SHAPE_BUCKET, HUC12_KEY)
    if gdf is not None:
        gdfs.append(gdf)

gdf_huc12 = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))

gdf_huc12.to_file('huc12_upload/huc12_boundaries.shp')

with zipfile.ZipFile('huc12_boundaries.zip', 'w') as zipf:
    for ext in ['.shp', '.shx', '.dbf', '.prj']:
        zipf.write(f'huc12_upload/huc12_boundaries{ext}', arcname=f'huc12_boundaries{ext}')

Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON
Successfully downloaded GeoJSON


Some HUC12s from the basins in the DHSVM data are not originally covered in the training set and are added separaately. We'll get boundaries for those as well.

In [3]:
gdf = gpd.read_file("huc12_upload/huc12_boundaries.shp")
dhsvm = pd.read_csv("dhsvm_lidar_hucs.csv")

# Normalize huc_id types
gdf["huc_id"] = gdf["huc_id"].astype(str)
dhsvm["huc_id"] = dhsvm["huc_id"].astype(str)

# Check coverage
boundary_hucs = set(gdf["huc_id"])
dhsvm_hucs = set(dhsvm["huc_id"])

in_boundaries = dhsvm_hucs & boundary_hucs
missing = list(dhsvm_hucs - boundary_hucs)

print(f"Total HUCs in dhsvm file:     {len(dhsvm_hucs)}")
print(f"Have boundaries for:          {len(in_boundaries)}")
print(f"Missing boundaries for:       {len(missing)}")
if missing:
    print(f"\nMissing HUC IDs:")
    for h in sorted(missing):
        print(f"  {h}")

Total HUCs in dhsvm file:     131
Have boundaries for:          40
Missing boundaries for:       91

Missing HUC IDs:
  171001010501
  171001010601
  171001010602
  171001010701
  171001010702
  171001010703
  171001010704
  171001010705
  171001020101
  171001020102
  171001020201
  171001020202
  171001020203
  171001020204
  171001020205
  171001020206
  171001020207
  171001020401
  171001020402
  171001020404
  171001020405
  171001020406
  171100100101
  171100100102
  171100100103
  171100100104
  171100100201
  171100100202
  171100100203
  171100100204
  171100100205
  171100100301
  171100100302
  171100100401
  171100100402
  171100100403
  171100100404
  171100100501
  171100100502
  171100100503
  171100100504
  171100100505
  171100100601
  171100100602
  171100100603
  171100110101
  171100110102
  171100110103
  171100110201
  171100110202
  171100110203
  171100110204
  171100120101
  171100120102
  171100120103
  171100120104
  171100120105
  171100120107
  1711001301

In [ ]:
# USGS NHD API — fetch in batches of 10 (URL length limit)
BASE_URL = "https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/6/query"

def fetch_hucs(huc_list):
    huc_filter = "','".join(huc_list)
    params = {
        "where": f"huc12 IN ('{huc_filter}')",
        "outFields": "huc12,name",
        "f": "geojson",
        "outSR": "4326"
    }
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    return response.json()

# Fetch in batches of 20
all_features = []
batch_size = 20
for i in range(0, len(missing), batch_size):
    batch = missing[i:i+batch_size]
    print(f"Fetching batch {i//batch_size + 1}...")
    geojson = fetch_hucs(batch)
    all_features.extend(geojson.get("features", []))

print(f"\nFetched {len(all_features)} features")

# Convert to GeoDataFrame
gdf_missing = gpd.GeoDataFrame.from_features(all_features, crs="EPSG:4326")
gdf_missing = gdf_missing.rename(columns={"huc12": "huc_id"})
gdf_missing["huc_id"] = gdf_missing["huc_id"].astype(str)

# Save
#gdf_missing.to_file("dhsvm_huc12_upload/dhsvm_huc12_boundaries_combined.shp")
gdf_missing.to_file("huc12_upload/huc12_boundaries_combined.shp")
print("Saved to huc12_upload/huc12_boundaries_combined.shp")

Then we upload these .shp files as GEE assets. 

In [ ]:
huc_fc = ee.FeatureCollection('projects/ee-riyoshas/assets/huc12_boundaries')
#huc_fc = ee.FeatureCollection('projects/ee-riyoshas/assets/dhsvm_huc12_boundaries')

## Yearwise Batch Export for SCF and SWE from ERA5 datatset
Now we fetch the ERA5 SCF and SWE data via yearwise batch exports

In [19]:
def map_over_collection(image):
    date = image.date().format('YYYY-MM-dd')
    stats = image.reduceRegions(
        collection=huc_fc,
        reducer=ee.Reducer.mean(),
        scale=11132
    )
    return stats.map(lambda f: f.set('date', date))

for year in range(1983, 2023):
    era5_year = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR') \
        .filterDate(f'{year}-10-01', f'{year+1}-10-01') \
        .select(['snow_cover', 'snow_depth_water_equivalent'])
    
    output = era5_year.map(map_over_collection).flatten()
    
    task = ee.batch.Export.table.toDrive(
        collection=output,
        description=f'dhsvm_ERA5_Snow_{year}',
        fileFormat='CSV',
        selectors=['date', 'huc_id', 'snow_cover', 'snow_depth_water_equivalent']
    )
    task.start()
    print(f"Started task for {year}")

Started task for 1983
Started task for 1984
Started task for 1985
Started task for 1986
Started task for 1987
Started task for 1988
Started task for 1989
Started task for 1990
Started task for 1991
Started task for 1992
Started task for 1993
Started task for 1994
Started task for 1995
Started task for 1996
Started task for 1997
Started task for 1998
Started task for 1999
Started task for 2000
Started task for 2001
Started task for 2002
Started task for 2003
Started task for 2004
Started task for 2005
Started task for 2006
Started task for 2007
Started task for 2008
Started task for 2009
Started task for 2010
Started task for 2011
Started task for 2012
Started task for 2013
Started task for 2014
Started task for 2015
Started task for 2016
Started task for 2017
Started task for 2018
Started task for 2019
Started task for 2020
Started task for 2021
Started task for 2022


The processed files will be saved in one's Google Drive. We'll transfer them to our S3 bucket.

In [ ]:
# Authenticate with Google Drive
gauth = GoogleAuth()
gauth.LocalWebserverAuth()
drive = GoogleDrive(gauth)

# S3 client
s3 = boto3.client("s3", region_name="us-west-2")

# Query for files matching pattern
file_list = drive.ListFile({"q": "trashed=false and name contains 'ERA5_Snow'"}).GetList()
#file_list = drive.ListFile({"q": "trashed=false and name contains 'dhsvm_ERA5_Snow'"}).GetList()

# Transfer files
for file in file_list:
    print(f"Downloading {file['title']}...")
    gdrive_file = drive.CreateFile({"id": file["id"]})
    
    # Download to memory
    file_content = io.BytesIO()
    gdrive_file.GetContentFile(file_content)
    file_content.seek(0)
    
    # Upload to S3
    s3_key = f"snow_cover_era5_land//{file['title']}"
    #s3_key = f"snow_cover_era5_land/dhsvm/{file['title']}"
    s3.upload_fileobj(file_content, "snowml-model-ready", s3_key)
    print(f"Uploaded {file['title']}")

# Get slope data 
This section uses Google Earth Engine to calculate mean slope for each HUC12 region using the USGS 3DEP 10m digital elevation model (DEM). 

In [ ]:
dem = ee.Image("USGS/3DEP/10m")
slope = ee.Terrain.slope(dem)

slope_stats = slope.reduceRegions(
    collection=huc_fc,
    reducer=ee.Reducer.mean(),
    scale=30
).select(['huc_id', 'mean'])

task = ee.batch.Export.table.toDrive(
    collection=slope_stats,
    #description='dhsvm_HUC12_Slope',
    description='HUC12_Slope',
    fileFormat='CSV',
    selectors=['huc_id', 'mean']
)
task.start()

/home/sagemaker-user/.conda/envs/snowml/lib/python3.11/site-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for USGS/3DEP/10m! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by USGS/3DEP/10m_collection

Learn more: https://developers.google.com/earth-engine/datasets/catalog/USGS_3DEP_10m

  warnings.warn(warning, category=DeprecationWarning)


Transfer the slope data from Google Drive to S3 bucket.

In [ ]:
slope_files = drive.ListFile({"q": "trashed=false and name contains 'HUC12_Slope'"}).GetList()
for file in slope_files:
    gdrive_file = drive.CreateFile({"id": file["id"]})
    file_content = io.BytesIO()
    gdrive_file.GetContentFile(file_content)
    file_content.seek(0)
    s3.upload_fileobj(file_content, "snowml-model-ready", f"slope_data/{file['title']}")
    print(f"{file['title']}")

In [ ]:
slope_df = pd.read_csv("HUC12_Slope.csv")
#slope_df = pd.read_csv("dhsvm_HUC12_Slope.csv")
slope_df["huc_id"] = slope_df["huc_id"].astype(str)
slope_df = slope_df.rename(columns={"mean": "slope"})[["huc_id", "slope"]]

# Add SCF and Slope data to Model Ready Bucket
This section loads ERA5 snow cover fraction (SCF), snow water equivalent (SWE) data, and mean slope values for each HUC12. It merges these variables into the model-ready HUC12 CSVs and uploads the enriched files to the S3 bucket for modeling and analysis.

In [10]:
def load_era5(bucket, prefix, years):
    """Load and concatenate ERA5 yearly CSVs into a single DataFrame."""
    s3 = boto3.client("s3", region_name='us-west-2')
    dfs = []
    for year in years:
        key = f"{prefix}dhsvm_ERA5_Snow_{year}.csv"
        try:
            body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
            if not body.strip():
                print(f"  [SKIP] {key} empty")
                continue
            dfs.append(pd.read_csv(io.BytesIO(body)))
            print(f"  [OK]   {key}")
        except Exception as e:
            print(f"  [ERR]  {key}: {e}")

    era5 = pd.concat(dfs, ignore_index=True)
    era5 = era5.rename(columns={
        "date": "day",
        "snow_depth_water_equivalent": "era5_swe"
    })
    era5["huc_id"] = era5["huc_id"].astype(str)
    era5["day"]    = era5["day"].astype(str)

    # Convert snow_cover from % to fraction
    era5["snow_cover"] = pd.to_numeric(era5["snow_cover"], errors="coerce") / 100.0
    era5["era5_swe"]   = pd.to_numeric(era5["era5_swe"],   errors="coerce")

    # Fill nulls with 0 (snow-free pixels)
    era5["snow_cover"] = era5["snow_cover"].fillna(0)
    era5["era5_swe"]   = era5["era5_swe"].fillna(0)

    era5 = era5[["huc_id", "day", "snow_cover", "era5_swe"]]
    print(f"\nERA5 loaded: {era5.shape}, HUCs: {era5['huc_id'].nunique()}")
    return era5


def list_huc_files(bucket):
    s3 = boto3.client("s3", region_name='us-west-2')
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket):
        for obj in page.get("Contents", []):
            k = obj["Key"]
            if k.startswith("model_ready_huc") and k.endswith(".csv") and obj["Size"] > 0:
                keys.append(k)
    return keys


def enrich_and_upload(huc_key, era5, slope_df):
    s3 = boto3.client("s3", region_name='us-west-2')
    body = s3.get_object(Bucket=SRC_BUCKET, Key=huc_key)["Body"].read()
    df = pd.read_csv(io.BytesIO(body))

    huc_id = huc_key.split("model_ready_huc")[1].split(".csv")[0]
    df["day"] = df["day"].astype(str)

    huc_era5 = era5[era5["huc_id"] == huc_id].drop(columns="huc_id")
    df = df.merge(huc_era5, on="day", how="left")

    # Add slope (static — same value for every row)
    slope_val = slope_df.loc[slope_df["huc_id"] == huc_id, "slope"]
    df["slope"] = slope_val.values[0] if len(slope_val) > 0 else None

    buf = io.BytesIO()
    df.to_csv(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=DST_BUCKET, Key=huc_key, Body=buf.getvalue())

In [ ]:
SRC_BUCKET    = "snowml-model-ready"
DST_BUCKET    = "snowml-model-ready-stgnn"
ERA5_PREFIX   = "snow_cover_era5_land/"
#ERA5_PREFIX   = "snow_cover_era5_land/dhsvm/"
YEARS         = range(1983, 2023)
REGION        = "us-west-2"
era5 = load_era5(SRC_BUCKET, ERA5_PREFIX, YEARS)

  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1983.csv
  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1984.csv
  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1985.csv
  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1986.csv
  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1987.csv
  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1988.csv
  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1989.csv
  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1990.csv
  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1991.csv
  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1992.csv
  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1993.csv
  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1994.csv
  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1995.csv
  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1996.csv
  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1997.csv
  [OK]   snow_cover_era5_land/dhsvm/dhsvm_ERA5_Snow_1998.csv
  [OK]   snow_cover_era5

In [13]:
print("\nListing HUC files...")
huc_keys = list_huc_files(SRC_BUCKET)
print(f"Found {len(huc_keys)} HUC files\n")

era5_hucs = set(era5["huc_id"].unique())
huc_keys = [k for k in huc_keys if k.split("model_ready_huc")[1].split(".csv")[0] in era5_hucs]
print(f"Filtered to {len(huc_keys)} HUC files with ERA5 coverage")


Listing HUC files...
Found 3681 HUC files

Filtered to 91 HUC files with ERA5 coverage


In [14]:
success, skipped = 0, 0
for i, key in enumerate(huc_keys):
    try:
        enrich_and_upload(key, era5, slope_df)
        success += 1
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(huc_keys)} processed...")
    except Exception as e:
        print(f"  [ERR] {key}: {e}")
        skipped += 1

print(f"\nDone. {success} uploaded, {skipped} failed → s3://{DST_BUCKET}/")

  50/91 processed...

Done. 91 uploaded, 0 failed → s3://snowml-model-ready-stgnn/
